# The scikit-learn API

**Author:** [firstname] [lastname]

COSC 3430 Artificial Intelligence | Fall 26 | Augustana U.

Estimators, transformers, predictors and five design principles. Code
along with the slides: each section below matches one code-along in the
slide deck.

Run the setup cell first. It loads the housing data and rebuilds
`housing`, `housing_labels` and `housing_num`.

In [ ]:
# Setup: load the housing data and rebuild the training set from parts 3 to 7
# data frames and arrays
import pandas as pd
import numpy as np
# train/test split
from sklearn.model_selection import train_test_split

# load the California housing data
housing_full = pd.read_csv("https://tinyurl.com/housing-csv")
# income categories for stratified sampling
housing_full["income_cat"] = pd.cut(
    housing_full["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5])
# 80/20 split that keeps the income mix in both sets
strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2,
    stratify=housing_full["income_cat"],
    random_state=42)
# drop the helper column from both sets
for df in (strat_train_set, strat_test_set):
    df.drop("income_cat", axis=1, inplace=True)
# predictors: the training set without the target
housing = strat_train_set.drop("median_house_value", axis=1)
# labels: a copy of the target column
housing_labels = strat_train_set["median_house_value"].copy()
# numerical features only
housing_num = housing.select_dtypes(include=[np.number])
# check the dimensions
print(housing.shape, housing_num.shape)

Expected output:

```
(16512, 9) (16512, 8)
```

## Code-along 1. An estimator: hyperparameters and learned parameters (demo)

In [ ]:
# Code-along 1: hyperparameter vs learned parameter
from sklearn.impute import SimpleImputer

# hyperparameter: set in the constructor
imputer = SimpleImputer(strategy="median")
print(imputer.strategy)
# learned parameter: not there before fit()
print(hasattr(imputer, "statistics_"))
imputer.fit(housing_num)
print(hasattr(imputer, "statistics_"))
# the medians it learned - compare with pandas
print(imputer.statistics_)
print(housing_num.median().values)

## Code-along 2. A transformer: fit on the training set only (demo)

In [ ]:
# Code-along 2: fit on the training set, transform both sets
from sklearn.preprocessing import StandardScaler

X_train = housing_num[["median_income"]]
X_test = strat_test_set[["median_income"]]

scaler = StandardScaler()
# learn mean and standard deviation of the training set
scaler.fit(X_train)
print(scaler.mean_, scaler.scale_)
# apply them to both sets
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(X_train_scaled.mean().round(3), X_train_scaled.std().round(3))
print(X_test_scaled.mean().round(3), X_test_scaled.std().round(3))

## Code-along 3. A predictor: `predict()` and `score()` (demo)

In [ ]:
# Code-along 3: a predictor - fit, predict, score
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
# supervised: features and labels
lin_reg.fit(X_train, housing_labels)
# learned parameters: slope and intercept
print(lin_reg.coef_, lin_reg.intercept_)
# predictions for the first three districts, and the true values
print(lin_reg.predict(X_train[:3]).round(-2))
print(housing_labels[:3].values)
# R^2 on the training set
print(round(lin_reg.score(X_train, housing_labels), 2))

## Code-along 4. Inspection (demo)

In [ ]:
# Code-along 4: look inside an estimator
# hyperparameters only
print(imputer.get_params())
# learned parameters: names that end with "_"
print([name for name in vars(imputer) if name.endswith("_")])

## Code-along 5. Sensible defaults (demo)

In [ ]:
# Code-along 5: what you get without arguments
from sklearn.neighbors import KNeighborsRegressor

print(KNeighborsRegressor().get_params())
print(SimpleImputer().get_params()["strategy"])

## Code-along 6. Non-proliferation of classes (demo)

In [ ]:
# Code-along 6: NumPy array in, NumPy array out
scaled = StandardScaler().fit_transform(housing_num)
print(type(scaled))
# ask for a DataFrame instead
scaler_pd = StandardScaler().set_output(transform="pandas")
scaled_df = scaler_pd.fit_transform(housing_num)
print(type(scaled_df))
print(scaled_df[["total_rooms", "median_income"]].head(3).round(2))

## Code-along 7. Composition: a pipeline is a predictor

## Practice: a class you have not used

`RobustScaler` (in `sklearn.preprocessing`) is new to you. Find out with
the API alone:

1. Which roles does it play?
   *Tip:* `hasattr(robust, "transform")`, the same for `"fit"` and `"predict"`.
2. What are its hyperparameters, and their defaults?
   *Tip:* `robust.get_params()`.
3. Fit it on `housing_num[["population"]]`. Which parameters did it learn,
   and what are their values?
   *Tip:* `vars(robust)` lists all attributes - the learned ones end with `_`.
4. Can you compute the same values with pandas?
   *Tip:* compare with `housing_num["population"].describe()`.